In [31]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from sksurv.ensemble import GradientBoostingSurvivalAnalysis
from category_encoders import TargetEncoder
from sklearn.ensemble import RandomForestClassifier
import warnings
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import roc_auc_score
from sksurv.metrics import brier_score
from sksurv.metrics import concordance_index_censored
from sksurv.nonparametric import kaplan_meier_estimator
from lifelines import KaplanMeierFitter
from sksurv.util import Surv
from xgboost import XGBClassifier
warnings.filterwarnings('ignore')

### Data preprocessing

In [32]:
# Load data
train = pd.read_csv('Data/train.csv')
test = pd.read_csv('Data/test.csv')
meta = pd.read_csv('Data/metaData.csv')

# Converting to dataframe
train = pd.DataFrame(train)
test = pd.DataFrame(test)

# Splitting train and test into X_train and y_train
X_train = train.drop(columns=['event', 'event_id', 'time_to_hit_hours'])
y_train = train[['time_to_hit_hours', 'event']]
y_train['event'] = y_train['event'].astype(int)

# y_train for gbsa
y_train_surv = Surv.from_arrays(event = y_train["event"].astype(bool), 
                                time = y_train["time_to_hit_hours"])
event_vals = y_train['event'].copy() # Later use for oof prediction loop
time_vals = y_train['time_to_hit_hours'].copy()

# Dropping event_id for X_test
X_test = test.drop(columns=['event_id'])
event_id = test['event_id'] # Save event_id for submitting

# Horizon
HORIZONS = [12, 24, 48, 72]
DROP_HORIZONS = [24, 48, 72]

# y_train for lightgbm (splitting each horizons)
y_train_lgbm_p12 = ((train['event'] == 1) & (train['time_to_hit_hours'] <= 12)).astype(int)
y_train_lgbm_p24 = ((train['event'] == 1) & (train['time_to_hit_hours'] <= 24)).astype(int)
y_train_lgbm_p48 = ((train['event'] == 1) & (train['time_to_hit_hours'] <= 48)).astype(int)
y_train_lgbm_p72 = ((train['event'] == 1) & (train['time_to_hit_hours'] <= 72)).astype(int)



In [33]:
# Helper function
def compute_c_index(time, event, risk):
    n = len(event)
    concordant = total = 0
    for i in range(n):
        if event[i] == 0: # Skip censored event
            continue
        for j in range(n):
            if i == j or time[i] >= time[j]: 
                continue
            total += 1
            if risk[i] == risk[j]:
                concordant += 0.5
            elif risk[i] > risk[j]:
                concordant += 1
    if total > 0:
        return concordant / total
    return 0.5

def compute_brier_sc(time, event, preds, horizons):
    brier_sc = np.zeros(len(horizons))
    for i, horizon in enumerate(horizons):
        uncensored_row = ~((time < horizon) & (event == 0))
        squared_error = (event[uncensored_row] - preds[uncensored_row, i])**2
        brier_sc[i] = np.sum(squared_error) / uncensored_row.sum()
    return brier_sc # Shape (4, )  

def compute_hybrid_sc(time, event, preds, horizons):
    risk_score = preds[:, 1] * 0.3 + preds[:, 2] * 0.4 + preds[:, 3] * 0.3
    c_idx = compute_c_index(time, event, risk_score)
    brier_score = compute_brier_sc(time, event, preds, horizons)
    p24 = brier_score[1]
    p48 = brier_score[2]
    p72 = brier_score[3]
    weighted_brier = 0.3*p24 + 0.4*p48 + 0.3*p72
    hybrid_score = 0.3*c_idx + 0.7*(1 - weighted_brier)
    return hybrid_score

def compute_ipcw():
    pass

### Gradient Boost Survival Analysis

In [34]:
# Total folds
N_FOLDS_GBSA = 5

# Initialize oof matrix prediction
oof_preds_gbsa = np.zeros((len(X_train), 4)) # 4 horizons: 12h, 24h, 48h, 72h

# Initialize final average test prediction
gbsa_test_preds = np.zeros((len(X_test), 4))

# Initialize 10 random seeds
GBSA_SEEDS = [42, 67, 69, 2026, 2006, 458, 478, 456, 782, 126]

# More config
gbsa_configs = [
    {"learning_rate":0.010, "subsample":0.70, "max_depth":3, "min_samples_leaf":12,  "min_samples_split":3, "n_estimators":1200},
    {"learning_rate":0.010, "subsample":0.85, "max_depth":3, "min_samples_leaf":15,  "min_samples_split":3, "n_estimators":1200},
    {"learning_rate":0.010, "subsample":0.60, "max_depth":3, "min_samples_leaf":12,  "min_samples_split":3, "n_estimators":1200},
    {"learning_rate":0.005, "subsample":0.85, "max_depth":3, "min_samples_leaf":12,  "min_samples_split":3, "n_estimators":2000},
    {"learning_rate":0.010, "subsample":0.85, "max_depth":3, "min_samples_leaf":20,  "min_samples_split":3, "n_estimators":1400},
    {"learning_rate":0.008, "subsample":0.75, "max_depth":2, "min_samples_leaf":15,  "min_samples_split":4, "n_estimators":1500},
    {"learning_rate":0.015, "subsample":0.70, "max_depth":3, "min_samples_leaf":10,  "min_samples_split":3, "n_estimators":1000},
    {"learning_rate":0.005, "subsample":0.90, "max_depth":3, "min_samples_leaf":18,  "min_samples_split":5, "n_estimators":2500},
    {"learning_rate":0.010, "subsample":0.80, "max_depth":4, "min_samples_leaf":12,  "min_samples_split":3, "n_estimators":1200},
    {"learning_rate":0.020, "subsample":0.65, "max_depth":3, "min_samples_leaf":10,  "min_samples_split":3, "n_estimators":800},
]

GBSA_SEEDS = [42, 67]
gbsa_configs = gbsa_configs[:2]

##### Train GBSA

In [35]:
for config in gbsa_configs:
    for seed in GBSA_SEEDS:
        skf = StratifiedKFold(n_splits=N_FOLDS_GBSA, shuffle=True, random_state=seed)
        seed_test_preds = np.zeros((len(X_test), len(HORIZONS)))

        for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, event_vals)):
            X_tr  = X_train.iloc[train_idx].copy()
            X_val = X_train.iloc[val_idx].copy()
            y_tr_surv  = y_train_surv[train_idx]

            # Initializing gbsa model
            gbsa = GradientBoostingSurvivalAnalysis(**config, random_state=seed)
            gbsa.fit(X_tr, y_tr_surv)

            # OOF
            val_preds_fold = np.zeros((len(val_idx), len(HORIZONS)))
            for i, sf in enumerate(gbsa.predict_survival_function(X_val)):
                for j, horizon in enumerate(HORIZONS):
                    val_preds_fold[i, j] = 1 - sf(min(horizon, sf.x[-1])) # type: ignore
            oof_preds_gbsa[val_idx] += val_preds_fold

            # Test predictions
            for i, sf in enumerate(gbsa.predict_survival_function(X_test)):
                for j, h in enumerate(HORIZONS):
                    seed_test_preds[i, j] += (1 - sf(min(h, sf.x[-1]))) # type: ignore

        # Average across folds for this (config, seed) pair
        gbsa_test_preds += seed_test_preds / N_FOLDS_GBSA

# Average across all (config, seed) pairs
total_runs = len(gbsa_configs) * len(GBSA_SEEDS)
oof_preds_gbsa  /= total_runs
gbsa_test_preds /= total_runs

oof_preds_gbsa  = np.maximum.accumulate(oof_preds_gbsa, axis=1)
gbsa_test_preds = np.maximum.accumulate(gbsa_test_preds, axis=1)

In [36]:
risk_score = oof_preds_gbsa[:, 1] * 0.3 + oof_preds_gbsa[:, 2] * 0.4 + oof_preds_gbsa[:, 3] * 0.3
risk_score.shape

(221,)

### GBSA evaluation (c_index, brier score)

In [37]:
c_index = compute_c_index(time_vals, event_vals, risk_score)
brier_score = compute_brier_sc(time_vals, event_vals, oof_preds_gbsa, HORIZONS)
hybrid_sc = compute_hybrid_sc(time_vals, event_vals, oof_preds_gbsa, HORIZONS)

print(f"C_index: {c_index:.4f}\n"
      f"Brier p24h: {brier_score[1]:.4f}\n"
      f"Brier p48h: {brier_score[2]:.4f}\n"
      f"Brier p72h: {brier_score[3]:.4f}\n"
      f"Hybrid score: {hybrid_sc:.4f}\n")

C_index: 0.9448
Brier p24h: 0.0079
Brier p48h: 0.0053
Brier p72h: 0.0010
Hybrid score: 0.9801



### Final submissions 

In [38]:
gbsa_test_preds[:, 3] = 1.0
submission = {
    'event_id' : event_id,
    'prob_12h' : gbsa_test_preds[:, 0],# type: ignore
    'prob_24h' : gbsa_test_preds[:, 1],# type: ignore
    'prob_48h' : gbsa_test_preds[:, 2],# type: ignore
    'prob_72h' : gbsa_test_preds[:, 3] # type: ignore
}
submission = pd.DataFrame(submission)
submission.to_csv('final_preds.csv', index=False)